### Fine Tuning dos Modelos com XGBoost

In [31]:
import json
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import numpy as np




In [13]:
# Carregar os dados
with open("../Resultados/Eduardo/10-Melhores-Info.json", "r", encoding="utf-8") as f:
    dados_json = json.load(f)

# Converter em DataFrame estruturado
linhas = []
for d in dados_json:
    hp = d["hiperparametros"]
    linhas.append({
        "posicao": d["posicao"],
        "activation": hp["activation"],
        "batch_size": hp["batch_size"],
        "learning_rate_init": hp["learning_rate_init"],
        "max_iter": hp["max_iter"],
        "n_iter_no_change": hp["n_iter_no_change"],
        "solver": hp["solver"],
        "hidden_layer_list": hp["hidden_layer_sizes"],
        "score": d["score_medio"]
    })

In [14]:
df = pd.DataFrame(linhas)
df

,posicao,activation,batch_size,learning_rate_init,max_iter,n_iter_no_change,solver,hidden_layer_list,score
0,1,relu,32,0.001,300,50,adam,"[4, 6]",0.973909
1,2,tanh,128,0.010,200,50,adam,[9],0.973338
2,3,relu,32,0.010,200,30,adam,[5],0.973223
3,4,relu,20,0.001,200,10,adam,"[4, 6]",0.972351
4,5,relu,100,0.010,300,30,adam,"[6, 4]",0.972183
5,6,relu,20,0.001,300,50,adam,[9],0.972041
6,7,relu,20,0.001,300,30,adam,"[3, 6]",0.971726
7,8,relu,32,0.001,200,10,adam,"[6, 4]",0.971569
8,9,relu,20,0.001,300,50,adam,"[6, 4]",0.971525
9,10,logistic,32,0.010,200,20,adam,[10],0.971348


In [15]:
# Transforma atributos em listas para atributos unicos
df["n_camadas"] = df["hidden_layer_list"].apply(len)
df["soma_neuronios"] = df["hidden_layer_list"].apply(sum)
df["max_neuronios"] = df["hidden_layer_list"].apply(max)
df["media_neuronios"] = df["hidden_layer_list"].apply(lambda x: sum(x)/len(x))

df_model = df.drop(columns=["hidden_layer_list", "posicao"])

In [16]:
X = df_model.drop(columns=["score"])
y = df_model["score"]

In [17]:
categoricas = ["activation", "solver"]
numericas = [c for c in X.columns if c not in categoricas]

preprocessador = ColumnTransformer([
    ("cat", OneHotEncoder(), categoricas),
    ("num", "passthrough", numericas)
])

In [24]:

modelo_xgb = XGBRegressor(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="rmse"
)


In [25]:
pipeline = Pipeline([
    ("prep", preprocessador),
    ("xgb", modelo_xgb)
])

pipeline.fit(X, y)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['activation', 'solver']),
                                                 ('num', 'passthrough',
                                                  ['batch_size',
                                                   'learning_rate_init',
                                                   'max_iter',
                                                   'n_iter_no_change',
                                                   'n_camadas',
                                                   'soma_neuronios',
                                                   'max_neuronios',
                                                   'media_neuronios'])])),
                ('xgb',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsa...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.03,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=5, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=400, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [26]:
# extrair nomes finais das features
nomes = pipeline.named_steps["prep"].get_feature_names_out()

# pegar importâncias do xgb
importancias = pipeline.named_steps["xgb"].feature_importances_

ranking = sorted(zip(nomes, importancias), key=lambda x: -x[1])

print("\n### IMPORTÂNCIA DOS HIPERPARÂMETROS ###")
for nome, peso in ranking:
    print(f"{nome}: {peso:.4f}")



### IMPORTÂNCIA DOS HIPERPARÂMETROS ###
num__batch_size: 0.3524
cat__activation_logistic: 0.3514
num__n_iter_no_change: 0.2963
cat__activation_relu: 0.0000
cat__activation_tanh: 0.0000
cat__solver_adam: 0.0000
num__learning_rate_init: 0.0000
num__max_iter: 0.0000
num__n_camadas: 0.0000
num__soma_neuronios: 0.0000
num__max_neuronios: 0.0000
num__media_neuronios: 0.0000


In [27]:
def gerar_candidato():
    return {
        "activation": np.random.choice(["relu", "tanh", "logistic"]),
        "batch_size": np.random.choice([20, 32, 64, 128]),
        "learning_rate_init": float(np.random.uniform(0.0005, 0.02)),
        "max_iter": np.random.choice([200, 300, 400]),
        "n_iter_no_change": np.random.choice([10, 20, 30, 50]),
        "solver": "adam",
        "n_camadas": np.random.choice([1, 2, 3]),
        "soma_neuronios": np.random.randint(8, 30),
        "max_neuronios": np.random.randint(4, 12),
        "media_neuronios": np.random.uniform(4, 10)
    }

def prever_score(c):
    temp = pd.DataFrame([c])  # precisa ser DataFrame
    return pipeline.predict(temp)[0]


In [28]:
def prever_score(c):
    temp = pd.DataFrame([c])  # precisa ser DataFrame
    return pipeline.predict(temp)[0]

In [32]:
candidates = []
for _ in range(200):
    c = gerar_candidato()
    s = prever_score(c)
    candidates.append((s, c))

melhores = sorted(candidates, key=lambda x: x[0], reverse=True)[:5]

print("\n### MELHORES NOVAS ARQUITETURAS PREVISTAS PELO XGBOOST ###")
for score_prev, cfg in melhores:
    print(f"Score previsto: {score_prev:.6f}  |  Hiperparâmetros: {cfg}")



### MELHORES NOVAS ARQUITETURAS PREVISTAS PELO XGBOOST ###
Score previsto: 0.972731  |  Hiperparâmetros: {'activation': 'relu', 'batch_size': 64, 'learning_rate_init': 0.0037740205698853133, 'max_iter': 400, 'n_iter_no_change': 50, 'solver': 'adam', 'n_camadas': 1, 'soma_neuronios': 11, 'max_neuronios': 6, 'media_neuronios': 9.108422927741728}
Score previsto: 0.972731  |  Hiperparâmetros: {'activation': 'tanh', 'batch_size': 64, 'learning_rate_init': 0.017115960867026016, 'max_iter': 200, 'n_iter_no_change': 50, 'solver': 'adam', 'n_camadas': 3, 'soma_neuronios': 19, 'max_neuronios': 5, 'media_neuronios': 7.740569300834673}
Score previsto: 0.972731  |  Hiperparâmetros: {'activation': 'tanh', 'batch_size': 128, 'learning_rate_init': 0.014975514436821512, 'max_iter': 300, 'n_iter_no_change': 50, 'solver': 'adam', 'n_camadas': 2, 'soma_neuronios': 8, 'max_neuronios': 10, 'media_neuronios': 9.67994991176967}
Score previsto: 0.972731  |  Hiperparâmetros: {'activation': 'tanh', 'batch_size'